Celda 1: Importación de librerías y carga de datos limpios

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

PATH_INPUT_LIMPIO = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_limpio.xlsx"
PATH_INPUT_MENSUAL = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_mensual.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset"

df_registro = pd.read_excel(PATH_INPUT_LIMPIO)
df_mensual = pd.read_excel(PATH_INPUT_MENSUAL)

print("Datos cargados para aumento de datos.")

Datos cargados para aumento de datos.


Celda 2: Aumento de Datos Mensuales (Bootstrapping con Ruido)

In [2]:
def augment_mensual_data(df, n_samples=120):
    cols_to_augment = ['servicios_totales', 'monto_total', 'carroza_count', 'auto_count', 'cargadores_total']
    
    df_augmented = resample(df, replace=True, n_samples=n_samples, random_state=42)
    
    for col in cols_to_augment:
        noise = np.random.normal(0, df[col].std() * 0.05, n_samples)
        df_augmented[col] = (df_augmented[col] + noise).clip(lower=0)
    
    df_augmented = df_augmented.sort_index().reset_index(drop=True)
    return df_augmented

df_mensual_aug = augment_mensual_data(df_mensual)
print(f"Dataset mensual aumentado: de {len(df_mensual)} a {len(df_mensual_aug)} registros.")

Dataset mensual aumentado: de 37 a 120 registros.


Celda 3: Balanceo de Categorías (SMOTE) para Inventario

In [3]:
le_modelo = LabelEncoder()
df_reg_smote = df_registro.copy()
df_reg_smote['Ataud_Modelo_Enc'] = le_modelo.fit_transform(df_reg_smote['Ataud_Modelo'])

X = df_reg_smote[['Monto', 'Carroza', 'Auto', 'Cargadores']]
y = df_reg_smote['Ataud_Modelo_Enc']

smote = SMOTE(sampling_strategy='auto', k_neighbors=1, random_state=42)
X_res, y_res = smote.fit_resample(X, y)

df_registro_aug = pd.DataFrame(X_res, columns=['Monto', 'Carroza', 'Auto', 'Cargadores'])
df_registro_aug['Ataud_Modelo'] = le_modelo.inverse_transform(y_res)

print(f"Dataset de registros aumentado con SMOTE: {len(df_registro_aug)} filas.")

ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 2, n_samples_fit = 1, n_samples = 1

Celda 3.5: Balanceo de Categorías (Random OverSampling)
Cambiamos SMOTE por esta

In [5]:
from imblearn.over_sampling import RandomOverSampler

df_reg_bal = df_registro.copy()

X = df_reg_bal[['Monto', 'Carroza', 'Auto', 'Cargadores', 'Ataud_Color', 'Capilla', 'Forma de pago', 'Ataud_completo']]
y = df_reg_bal['Ataud_Modelo']

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

df_registro_aug = pd.DataFrame(X_res)
df_registro_aug['Ataud_Modelo'] = y_res

print(f"Dataset de registros balanceado exitosamente.")
print(f"Total registros antes: {len(df_registro)} -> Después: {len(df_registro_aug)}")
print("\nConteo por modelo de ataúd en el nuevo dataset:")
print(df_registro_aug['Ataud_Modelo'].value_counts())

Dataset de registros balanceado exitosamente.
Total registros antes: 14 -> Después: 27

Conteo por modelo de ataúd en el nuevo dataset:
Ataud_Modelo
No_Especificado      3
Americano            3
Lincoln Biblia       3
Redondo              3
Foroon               3
Modelo               3
Biblia               3
Biblia Panoramico    3
Farana               3
Name: count, dtype: int64


Celda 4: Ingeniería de Detalles

In [6]:
df_mensual_aug['monto_mes_anterior'] = df_mensual_aug['monto_total'].shift(1).fillna(0)
df_mensual_aug['servicios_mes_anterior'] = df_mensual_aug['servicios_totales'].shift(1).fillna(0)

df_mensual_aug['tendencia_3m'] = df_mensual_aug['servicios_totales'].rolling(window=3).mean().fillna(0)

print("Variables de ingeniería (Lags y Medias Móviles) creadas.")

Variables de ingeniería (Lags y Medias Móviles) creadas.


Celda 5: Exportación del Dataset de Entrenamiento

In [7]:
df_registro_aug.to_excel(os.path.join(PATH_OUTPUT_DIR, "train_registro_aug.xlsx"), index=False)
df_mensual_aug.to_excel(os.path.join(PATH_OUTPUT_DIR, "train_mensual_aug.xlsx"), index=False)

reporte_aug = pd.DataFrame({
    'Dataset': ['Mensual Aumentado', 'Registros SMOTE'],
    'Filas Originales': [len(df_mensual), len(df_registro)],
    'Filas Finales': [len(df_mensual_aug), len(df_registro_aug)]
})
reporte_aug.to_excel(os.path.join(PATH_OUTPUT_DIR, "metricas_augmentation.xlsx"), index=False)

print("Notebook 2 finalizado. Datos listos para el entrenamiento de modelos.")

Notebook 2 finalizado. Datos listos para el entrenamiento de modelos.
